In [ ]:
# Imports & Device


import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Re-define Secure-CNN Architecture

import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)


class SecureCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.res1 = ResidualBlock(128)
        self.res2 = ResidualBlock(128)
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

       
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.res1(x)
        x = self.res2(x)
        x = self.conv3(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


In [ ]:
# Load Trained Secure-CNN Weights

secure_model = SecureCNN().to(device)

secure_model.load_state_dict(
    torch.load("secure_cnn_epoch30.pth", map_location=device)
)

secure_model.eval()
print("Secure-CNN loaded successfully")

In [ ]:
# Load CIFAR-10 Test Data

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4
)

In [ ]:
# Clean Accuracy Evaluation

def evaluate_clean(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total

In [ ]:
clean_acc = evaluate_clean(secure_model, test_loader)
print(f"Secure-CNN Clean Accuracy: {clean_acc:.2f}%")


# FGSM & PGD Attacks

In [ ]:
criterion = nn.CrossEntropyLoss()


In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    images.requires_grad = True
    outputs = model(images)
    loss = criterion(outputs, labels)
    model.zero_grad()
    loss.backward()
    adv_images = images + epsilon * images.grad.sign()
    return torch.clamp(adv_images, -1, 1)


In [ ]:
def pgd_attack(model, images, labels, epsilon, alpha=0.007, steps=10):
    ori_images = images.clone()
    adv_images = images.clone()

    for _ in range(steps):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = criterion(outputs, labels)
        model.zero_grad()
        loss.backward()
        adv_images = adv_images + alpha * adv_images.grad.sign()
        eta = torch.clamp(adv_images - ori_images, -epsilon, epsilon)
        adv_images = torch.clamp(ori_images + eta, -1, 1).detach()

    return adv_images


In [ ]:
# Evaluate Under FGSM & PGD

def evaluate_attack(model, dataloader, attack_fn, epsilon):
    model.eval()
    correct = 0
    total = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        adv_images = attack_fn(model, images, labels, epsilon)
        outputs = model(adv_images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return 100 * correct / total

In [ ]:
fgsm_acc = evaluate_attack(secure_model, test_loader, fgsm_attack, epsilon=0.03)
pgd_acc  = evaluate_attack(secure_model, test_loader, pgd_attack, epsilon=0.03)

print(f"FGSM Accuracy (ε=0.03): {fgsm_acc:.2f}%")
print(f"PGD  Accuracy (ε=0.03): {pgd_acc:.2f}%")


In [ ]:
pgd_acc  = evaluate_attack(secure_model, test_loader, pgd_attack, epsilon=0.05)
print(f"PGD  Accuracy (ε=0.05): {pgd_acc:.2f}%")
